# Border-cropped APTOS preprocessing

Crops dark outer margins from APTOS images using a fixed grayscale threshold, then resizes each cropped image to 224 × 224. Labels are unchanged.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,cv2,shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
BASE_DIR='/content/drive/MyDrive/Fundus_Artifact_Project'
APTOS_IMG_DIR=os.path.join(BASE_DIR,'APTOS_2019','train_images')
APTOS_CSV=os.path.join(BASE_DIR,'APTOS_2019','train.csv')
SAVE_DIR=os.path.join(BASE_DIR,'Results','dr_artifact_model')
CLEAN_IMG_DIR=os.path.join(SAVE_DIR,'cleaned_images')
BINARY_DIR=os.path.join(SAVE_DIR,'binary_dataset')
os.makedirs(CLEAN_IMG_DIR,exist_ok=True)


In [ ]:
def crop_black_borders(img, threshold=15):
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    _,mask=cv2.threshold(gray,threshold,255,cv2.THRESH_BINARY)
    coords=cv2.findNonZero(mask)
    if coords is None: return img
    x,y,w,h=cv2.boundingRect(coords)
    return img[y:y+h,x:x+w]

df=pd.read_csv(APTOS_CSV)
for _,row in tqdm(df.iterrows(),total=len(df)):
    src=os.path.join(APTOS_IMG_DIR,row['id_code']+'.png')
    dst=os.path.join(CLEAN_IMG_DIR,row['id_code']+'.png')
    if not os.path.isfile(src): continue
    img=cv2.imread(src)
    clean=crop_black_borders(img)
    clean=cv2.resize(clean,(224,224),interpolation=cv2.INTER_AREA)
    cv2.imwrite(dst,clean)


In [ ]:
# Build ImageFolder-style binary dataset
for label in ['dr','no_dr']: os.makedirs(os.path.join(BINARY_DIR,label),exist_ok=True)
for _,row in tqdm(df.iterrows(),total=len(df)):
    src=os.path.join(CLEAN_IMG_DIR,row['id_code']+'.png')
    if not os.path.isfile(src): continue
    label='no_dr' if int(row['diagnosis'])==0 else 'dr'
    shutil.copy2(src,os.path.join(BINARY_DIR,label,row['id_code']+'.png'))


In [ ]:
# Inspect a few processed images
import random
import matplotlib.pyplot as plt
from PIL import Image
fig,axes=plt.subplots(2,4,figsize=(14,7))
for r,label in enumerate(['no_dr','dr']):
    folder=os.path.join(BINARY_DIR,label)
    samples=random.sample(os.listdir(folder),min(4,len(os.listdir(folder))))
    for c,name in enumerate(samples):
        axes[r,c].imshow(Image.open(os.path.join(folder,name))); axes[r,c].axis('off'); axes[r,c].set_title(label)
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR,'sanity_check_cleaned_samples.png'),dpi=300,bbox_inches='tight'); plt.show()


## Note

This crop is intentionally simple. The later experiments test its effect rather than treating it as an optimal retinal preprocessing method.
